NAMES: IRAKOZE Furaha
PROGRAM: BIG DATA ANALYTICS
LAB 6

In [ ]:
import os, json, sqlite3, warnings
import pandas as pd
import requests
warnings.filterwarnings("ignore")

In [5]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\HP\BigDataLab
['.ipynb_checkpoints', 'BigDataCourse', 'class_dashboard.html', 'Untitled.ipynb', 'venv', 'Week10_IRAKOZEFuraha.ipynb', 'week10_students.csv', 'week12_transactions.csv', 'Week1_IRAKOZEFuraha.ipynb', 'week3_IRAKOZE_Furaha.ipynb', 'week3_report.txt', 'week3_students.csv', 'week5_IRAKOZEFuraha.ipynb', 'week5_kigali_weather (1).csv', 'Week6_IRAKOZEFuraha.ipynb']


In [7]:
df = pd.read_csv("week12_transactions.csv")

In [8]:
import sqlite3
import pandas as pd

df = pd.read_csv("week12_transactions.csv")
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
print(df.shape)   # (2000, 7)

(2000, 7)


Part 1: SQL
Exercise 1.1 — SELECT / WHERE / ORDER BY

In [10]:
q = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC, txn_id
LIMIT 5;
"""
pd.read_sql(q, conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


1.1 (b)


In [11]:
pd.read_sql("SELECT COUNT(*) AS n_flagged FROM transactions WHERE flagged = 1", conn)

,n_flagged
0,11


1.2 (a)

In [14]:
q = """
SELECT   txn_type,
         COUNT(*)        AS n_txns,
         SUM(amount_rwf) AS total_rwf,
         AVG(amount_rwf) AS avg_rwf
FROM     transactions
GROUP BY txn_type
ORDER BY total_rwf DESC;
"""
pd.read_sql(q, conn)

,txn_type,n_txns,total_rwf,avg_rwf
0,send,868,62049000,71485.023041
1,paybill,330,29970000,90818.181818
2,receive,489,27024000,55263.803681
3,airtime,313,833000,2661.341853


**Answer:** `send` moves the most money, about **62.0M RWF** (868 transactions).

Exercise 1.2 — GROUP BY

In [15]:
q = """
SELECT   district,
         SUM(amount_rwf) AS total_rwf
FROM     transactions
GROUP BY district
ORDER BY total_rwf DESC;
"""
pd.read_sql(q, conn)

,district,total_rwf
0,Muhanga,21076000
1,Gasabo,20738000
2,Huye,18736000
3,Nyarugenge,18091000
4,Kicukiro,11338000
5,Musanze,10490500
6,Rubavu,10110500
7,Nyagatare,9296000


Answer: Muhanga leads with about 21.1M RWF.
 
  
Exercise 1.3 — The busiest hour

In [16]:
# (a) SQL
q = "SELECT hour, COUNT(*) AS n_txns FROM transactions GROUP BY hour ORDER BY n_txns DESC;"
sql_hours = pd.read_sql(q, conn)

# (b) Pandas
pd_hours = df.groupby("hour").size().sort_values(ascending=False)

print(sql_hours.iloc[0, 0] == pd_hours.index[0], sql_hours.iloc[0, 1] == pd_hours.iloc[0])  # True True

True True


Answer: Hour 20 is busiest with 185 transactions. Hour 17 has 183 and hour 18 has 176.

2.1 JSON by hand 

In [ ]:
# NOTE: if the lab notebook gave you a JSON string, paste it here instead of this sample.
json_string = '{"city": "Kigali", "readings": [21.5, 23.0, 25.5, 27.0, 24.0]}'

obj = json.loads(json_string)
print("(a) City              :", obj["city"])
print("(b) Number of readings:", len(obj["readings"]))
print("(c) Average           :", round(sum(obj["readings"]) / len(obj["readings"]), 2))

(a) City              : Kigali
(b) Number of readings: 5
(c) Average           : 24.2


Exercise 2.2 — Call a real API


In [ ]:
url = ("https://api.open-meteo.com/v1/forecast"
       "?latitude=-1.95&longitude=30.06&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
try:
    data = requests.get(url, timeout=10).json()
    print("Live API call succeeded")
except Exception as e:
    print("API unavailable, using offline sample:", e)
    with open("week12_api_sample.json") as f:
        data = json.load(f)

In [25]:
times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("(a) Hours:", len(temps))
print("(b) Max:", max(temps), "°C   Min:", min(temps), "°C")
print("(c) Hottest hour at:", times[temps.index(max(temps))])

(a) Hours: 24
(b) Max: 24.5 °C   Min: 15.6 °C
(c) Hottest hour at: 2026-11-24T11:00


Exercise 2.3 — API data meets your toolbox

In [26]:
weather = pd.DataFrame({"time": times, "temp_c": temps})
print(weather["temp_c"].describe())                        # (a)
print("Hours >= 25 °C:", (weather["temp_c"] >= 25).sum())  # (b)

count    24.000000
mean     18.650000
std       2.507554
min      15.600000
25%      16.525000
50%      18.100000
75%      20.300000
max      24.500000
Name: temp_c, dtype: float64
Hours >= 25 °C: 0


Part 3 · PySpark 



Exercise 3.1 — Load & inspect

In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("momo").getOrCreate()

sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)
print(sdf.count())      # should print 2000
sdf.printSchema()
sdf.show(5)

2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


3.2 FILTER

In [4]:
# (a) transactions above 100,000 RWF
print("Above 100k:", sdf.filter(sdf.amount_rwf > 100000).count())   # Checkpoint: 451

# (b) 5 flagged transactions
sdf.filter(sdf.flagged == 1).show(5)

Above 100k: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


3.3 — groupBy, fourth reunion 


In [17]:
sdf.groupBy("txn_type").sum("amount_rwf").show()                       # (a)

by_district = (sdf.groupBy("district").sum("amount_rwf").toPandas()
                  .sort_values("sum(amount_rwf)", ascending=False))     # (b)
print(by_district)


# (c) Yes, the Spark answers match SQL: send = 62,049,000 RWF is the largest type,
#     and Muhanga = 21,076,000 RWF leads the districts. Same GROUP BY idea, different engine.

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+

     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


In [18]:
sdf.createOrReplaceTempView("transactions")

spark.sql("""
SELECT txn_type,
       COUNT(*)         AS n_txns,
       SUM(amount_rwf)  AS total_rwf,
       AVG(amount_rwf)  AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
""").show()

# Why is it powerful that the same SQL runs on SQLite and on a thousand-machine cluster?
# Because you learn ONE language and reuse it everywhere: prototype on a small sample on your
# laptop with SQLite, then run the unchanged query on a cluster holding terabytes. No rewriting,
# no new skills, and analysts, engineers and tools all share a common, portable language,
# while Spark handles the distribution of the work across machines.

+--------+------+---------+------------------+
|txn_type|n_txns|total_rwf|           avg_rwf|
+--------+------+---------+------------------+
|    send|   868| 62049000| 71485.02304147465|
| paybill|   330| 29970000| 90818.18181818182|
| receive|   489| 27024000|  55263.8036809816|
| airtime|   313|   833000|2661.3418530351437|
+--------+------+---------+------------------+



 Part 4 · Choose Your Weapon 

1. **SQL** — the data already lives in a PostgreSQL database, so the first move is to run a `GROUP BY` query (monthly totals per district) right where the data is, instead of exporting it.
2. **Pandas** — 40,000 rows fit comfortably in memory on one laptop, and Pandas handles cleaning, plotting and feeding a regression model (with scikit-learn) in one place.
3. **Spark** — 40 TB is far too big for one machine, but it is already on the telecom's cluster, so Spark can process it in parallel where it sits.
4. **API** — only a live API (e.g. Open-Meteo) delivers fresh weather automatically on each request, with nobody needing to upload a file.

# Part 5 · Final-Project Proposal

**Question:** Can we predict the value of a mobile-money transaction from when and where it happens, and spot unusual ones? **Dataset:** [group's dataset: source and size, minimum 500 rows, e.g. Rwanda mobile-money transactions with type, district, hour, amount and fraud flag]. **Why it matters:** Mobile money is central to daily payments in Rwanda; knowing peak hours, high-value districts and suspicious patterns helps operators plan capacity and detect fraud early. **Planned model:** A regression model (e.g. linear regression) to predict `amount_rwf` from hour, district and transaction type, because the target is a continuous number; if the group focuses on fraud we would use classification (e.g. logistic regression) on the `flagged` label.